# Build SNP Nakeds. 
 - One Put Option per symbol

In [1]:
## THIS CELL SHOULD BE IN ALL VSCODE NOTEBOOKS ##

MARKET = "SNP"

# Set the root
from from_root import from_root # type: ignore
ROOT = from_root()

import pandas as pd # type: ignore
from loguru import logger # type: ignore

pd.options.display.max_columns = None
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

from pathlib import Path
import sys

# Add `src` and ROOT to _src.pth in .venv to allow imports in VS Code
from sysconfig import get_path

if "src" not in Path.cwd().parts:
    src_path = str(Path(get_path("purelib")) / "_src.pth")
    with open(src_path, "w") as f:
        f.write(str(ROOT / "src\n"))
        f.write(str(ROOT))
        if str(ROOT) not in sys.path:
            sys.path.insert(1, str(ROOT))

# Start the Jupyter loop
from ib_async import util # type: ignore

util.startLoop()

logger.add(sink=ROOT / "log" / "ztest.log", mode="w")

1

In [2]:
import itertools

import numpy as np
from ib_async import util

from ibfuncs import (MarketOrder, get_ib, get_option_chains, make_chains,
                     marginsAsync, qualify_me)
from snp import assemble_snp_underlyings, us_repo_rate
from utils import (append_safe_strikes, convert_to_utc_datetime,
                   get_closest_strike, get_dte, get_pickle, how_many_days_old,
                   load_config)
from utils import append_black_scholes
from ib_async import Contract
from utils import clean_ib_util_df

In [3]:
# Set constants

config = load_config(MARKET=MARKET)

MAXDTE = config.get("MAXDTE")
PUTSTDMULT = config.get("PUTSTDMULT")
CALLSTDMULT = config.get("CALLSTDMULT")
MINEXPROM = config.get("MINEXPROM")

In [4]:
# delete logs
from utils import overwrite_logs
# files = ROOT/'log'/'ztest.log'
overwrite_logs()

# Make unds

In [5]:
df_unds = assemble_snp_underlyings(FRESH=False)

In [6]:
# assemble the underlyings
unds_path = ROOT / 'data' / 'snp_unds.pkl'
if how_many_days_old(unds_path) > 1:
    df_unds = assemble_snp_underlyings(FRESH=True)
else:
    df_unds = get_pickle(unds_path)
df_unds.head()

,ib_symbol,undId,secType,expiry,strike,right,contract,undPrice,und_iv,und_hv
0,MMM,9720,STK,NaT,0.00,,"Stock(conId=9720, symbol='MMM', exchange='SMAR...",131.08,0.19,0.39
1,ABT,4065,STK,NaT,0.00,,"Stock(conId=4065, symbol='ABT', exchange='SMAR...",112.24,0.17,0.29
2,ABBV,118089500,STK,NaT,0.00,,"Stock(conId=118089500, symbol='ABBV', exchange...",196.60,0.18,0.25
3,ACN,67889930,STK,NaT,0.00,,"Stock(conId=67889930, symbol='ACN', exchange='...",333.87,0.25,0.22
4,ADBE,265768,STK,NaT,0.00,,"Stock(conId=265768, symbol='ADBE', exchange='S...",558.40,0.41,0.28


# Make chains from the unds

In [7]:
# make / get the chains

opts_path = ROOT/'data/'/'snp_opts.pkl'
if how_many_days_old(opts_path) > 1:
    df_ch = make_chains(df_unds, save=True)
else:
    df_ch = get_pickle(opts_path)

df_ch.groupby('ib_symbol').first().head()

,undId,secType,undPrice,und_iv,und_hv,expiry,strike,dte,right
ib_symbol,,,,,,,,,
AAL,139673266,STK,10.40,0.41,0.44,2024-08-30 10:00:00+00:00,1.00,3.42,P
AAPL,265598,STK,226.38,0.22,0.42,2024-08-30 10:00:00+00:00,5.00,3.42,P
ABBV,118089500,STK,196.60,0.18,0.25,2024-08-30 10:00:00+00:00,70.00,3.42,P
ABNB,459530964,STK,116.99,0.28,0.54,2024-08-30 10:00:00+00:00,45.00,3.42,P
ABT,4065,STK,112.24,0.17,0.29,2024-08-30 10:00:00+00:00,50.00,3.42,P


# Make PUT targets from chains - closest to strike

In [8]:
dfp = df_ch[df_ch.right == "P"]
dfp = dfp[dfp.dte <= MAXDTE]

dfe = dfp[dfp.groupby(['ib_symbol', 'dte']).dte.transform('min').astype('int') == dfp.dte.astype('int')]

In [9]:
dft = dfe.groupby(['ib_symbol', 'dte']) \
        .apply(lambda x: get_closest_strike(x), include_groups=False) \
        .reset_index().set_index('level_2') \
        .rename_axis('')

dft = dft.sort_values(['ib_symbol', 'dte'])

# Compute the IV to be average of historical and implied, if implied is less than historical

min_series = pd.Series(np.minimum(dft.und_iv, dft.und_hv))
weighted_avg_series = (dft.und_iv + dft.und_hv) / 2 * 0.75
iv = pd.Series(np.where(dft.und_iv < dft.und_hv, min_series + weighted_avg_series, dft.und_iv), index=dft.index)
dft = dft.assign(iv=iv)

# Get the safe strikes
dft = append_safe_strikes(dft, PUTSTDMULT, CALLSTDMULT)

# Make xPrice from black-scholes and market price

In [10]:
# Get black scholes price

risk_free_rate = us_repo_rate() / 100
dft = append_black_scholes(dft, risk_free_rate)

In [11]:
dft.groupby('ib_symbol').first().head()

,dte,undId,secType,undPrice,und_iv,und_hv,expiry,strike,right,iv,sdev,safe_strike,intrinsic,bsPrice
ib_symbol,,,,,,,,,,,,,,
AAL,3.42,139673266,STK,10.40,0.41,0.44,2024-08-30 10:00:00+00:00,15.00,P,0.73,0.73,9,6.00,4.59
AAPL,3.42,265598,STK,226.38,0.22,0.42,2024-08-30 10:00:00+00:00,365.00,P,0.46,9.99,209,156.00,138.43
ABBV,3.42,118089500,STK,196.60,0.18,0.25,2024-08-30 10:00:00+00:00,230.00,P,0.34,6.46,185,45.00,33.28
ABNB,3.42,459530964,STK,116.99,0.28,0.54,2024-08-30 10:00:00+00:00,225.00,P,0.59,6.71,105,120.00,107.89
ABT,3.42,4065,STK,112.24,0.17,0.29,2024-08-30 10:00:00+00:00,118.00,P,0.34,3.66,106,12.00,5.80


In [12]:
# Get the market price with margin and commissions

contracts = [Contract('OPT', symbol=s, lastTradeDateOrContractMonth=util.formatIBDatetime(e)[:8], strike=k, right=r, 
                    exchange='SMART', currency='USD') 
                    for s, e, k, r 
                    in zip(dft.ib_symbol, dft.expiry, dft.strike, dft.right)]




In [13]:
from ib_async import IB
from tqdm import tqdm

from utils import chunk_me


async def process_in_chunks(ib: IB, data: any, 
                            chunk_size: int = 25, 
                            func: callable = None, 
                            func_args: dict = None, 
                            chunk_desc: str = "Processing data"):
    """
    Process data in chunks using the provided function.

    Args:
        ib (IB): An instance of the IB class.
        data (list | pd.DataFrame): The data to be processed.
        chunk_size (int, optional): The size of each chunk. Defaults to 25.
        func (callable, optional): The function to be used for processing the data. Defaults to None.
        func_args (dict, optional): The arguments to be passed to the function. Defaults to None.
        desc (str, optional): The description to be used in the tqdm progress bar. Defaults to "Processing data".

    Returns:
        list: The processed data.
    """
    if not func:
        raise ValueError("A function must be provided for processing the data.")

    if not func_args:
        func_args = {}

    chunks = chunk_me(data, chunk_size)

    processed_data = []

    for chunk in tqdm(chunks, desc=chunk_desc):
        func_args["data"] = chunk
        processed_chunk = await func(ib, **func_args)
        processed_data.extend(processed_chunk)

    return processed_data

In [14]:
contracts = contracts[:48] # !!! TEMPORARY LIMIT
with get_ib(MARKET) as ib:
    cts = ib.run(process_in_chunks(ib=ib, data=contracts, func=qualify_me, chunk_size=44, chunk_desc='qualifying chunks', func_args={'desc': '...contracts'}))

qualifying chunks: 100%|██████████| 2/2 [00:02<00:00,  1.20s/it]


In [15]:
from ibfuncs import get_mkt_prices
with get_ib(MARKET) as ib:
    prices = ib.run(process_in_chunks(ib, cts, chunk_size=35, func=get_mkt_prices, func_args={'gentick': '104,106', 'sleep': 7}))


Processing data: 100%|██████████| 1/1 [00:07<00:00,  7.41s/it]


In [16]:
prices

['ib_symbol',
 'conId',
 'secType',
 'expiry',
 'strike',
 'right',
 'contract',
 'price',
 'iv',
 'hv']

In [ ]:
orders = [MarketOrder('SELL', 1) for _ in cts]

In [ ]:


dft = dft.assign(contract = contracts, order = orders)


In [ ]:
with get_ib(MARKET) as ib:
    contracts = ib.run(qualify_me(ib, contracts))


In [ ]:
dfclean = clean_ib_util_df([c for c in contracts if c])
orders = [MarketOrder('SELL', 1) for _ in dfclean.contract]
dfclean = dfclean.assign(order=orders)

In [ ]:
from ib_async import IB

from ibfuncs import get_one_margin, margin_comm
from tqdm import tqdm
import asyncio

from utils import chunk_me

async def marginsAsync(
    ib: IB, df: pd.DataFrame,
    timeout: float = 2,
    chunk_size: int = 25,
    ) -> pd.DataFrame:
    """Gets async contracts from a df

    Args:
        ib (IB): An active IB connection
        df (pd.DataFrame): df with `contract` and `order` fields
        timeout (float, optional): time delay to get a margin. Defaults to 2.
        chunk_size (int, optional): size of each chunk for processing. Defaults to 25.
        eod (bool, optional): gets end of day time for options. Defaults to True.
        ist (bool, optional): gets Indian Std Time for NSE options. Defaults to True.

    Returns:
        pd.DataFrame: df with ib_symbol, margin and comm
    """
    try:
        contracts = df.contract.to_list()
        orders = df.order.to_list()
    except ValueError as e:
        logger.error(f"df does not have contract or order.Error: {e}")
        return pd.DataFrame([])

    # qualify contracts if there is no conId
    if df.contract.iloc[0].conId == 0:
        await ib.qualifyContractsAsync(*contracts)

    cos = list(zip(contracts, orders))

    tasks_chunks = chunk_me(cos, chunk_size)

    results = []
    for chunk in tqdm(tasks_chunks):
        chunk_tasks = [asyncio.create_task(get_one_margin(ib, c, o, timeout)) for c, o in chunk]
        chunk_results = await asyncio.gather(*chunk_tasks)
        results.extend(chunk_results)

    mcom = [margin_comm(r) for r in results]

    # df1 = pd.DataFrame(mcom, columns=["margin", "comm"])
    # df_mcom = df1.assign(contract=contracts)

    return mcom

In [ ]:
with get_ib(MARKET) as ib:    
    df_mcom = ib.run(marginsAsync(ib=ib, df=dfclean), timeout=10)


# Get margins

In [ ]:
# make orders
orders = [MarketOrder('SELL', 100) for _ in df_unds.contract]
dfco = df_unds.assign(order=orders)

In [ ]:
# get the margins
with get_ib(MARKET) as ib:
    df_mcom = ib.run(marginsAsync(ib=ib, df=dfco, timeout=10))

In [ ]:
df_unds = df_unds.assign(contract=df_mcom.contract, margin=df_mcom.margin, comm=df_mcom.comm)
iv = ((df_unds.und_iv+df_unds.und_hv)/2)
df_unds.insert(10, 'iv', iv )

In [ ]:
df_ch.head()

# Find closest strike margins (if possible)

In [ ]:
# Get closest and earliest strikes
dfe = df_ch[df_ch.groupby('ib_symbol').dte.transform('min').astype('int') == df_ch.dte.astype('int')]
dfep = dfe[dfe.right == 'P']
dfce = dfep.groupby(['ib_symbol', 'dte']).apply(lambda x: get_closest_strike(x), include_groups=False)

In [ ]:
dfce

In [ ]:
dfu = dfce.reset_index().set_index('level_2').rename_axis('')
dfu = dfu.sort_values('undPrice', ascending=False).groupby('ib_symbol').head(1)
cols = ['ib_symbol', 'expiry', 'dte', 'strike', 'undPrice','right']
dfu = dfu[cols].assign(action='SELL')

In [ ]:
from ibfuncs import qualify_me, get_ib
from ib_async import MarketOrder, Contract, util

contracts = [Contract('OPT', symbol=s, lastTradeDateOrContractMonth=util.formatIBDatetime(e)[:8], strike=k, right=r, 
                    exchange='SMART', currency='USD') 
                    for s, e, k, r 
                    in zip(dfu.ib_symbol, dfu.expiry, dfu.strike, dfu.right)]

orders = [MarketOrder('SELL', 1) for _ in contracts]

dfu = dfu.assign(contract = contracts, order = orders)

In [ ]:
from ibfuncs import marginsAsync


with get_ib(MARKET) as ib:
    
    contracts = ib.run(qualify_me(ib, contracts))
    df_mcom = ib.run(marginsAsync(ib=ib, df=dfu, timeout=10))
    

In [ ]:
contracts[0]

In [ ]:
dfu[dfu.ib_symbol == 'BKNG']